In [2]:
"""
=============================================================
DSA COMPFEST 18 — Best Submission (LB Score: 12.37054)
=============================================================
Model    : LightGBM only
CV       : 5-Fold KFold (standard)
Target   : log1p transform
FE       : Missing indicators + kimia tanah + spektral +
           OOF target encoding (source_id, biome)
=============================================================
"""

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# ============================================================
# KONFIGURASI
# ============================================================
SEED    = 42
N_FOLDS = 5
TARGET  = 'property_organic_content'

np.random.seed(SEED)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# ============================================================
# LOAD DATA
# ============================================================
DATA_DIR = '/kaggle/input/competitions/seleksi-data-science-academy-compfest-18'
OUT_DIR  = '/kaggle/working/'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')

train_ids = train['sample_id'].copy()
test_ids  = test['sample_id'].copy()

# Simpan kolom mentah untuk target encoding (sebelum fe)
source_col_raw = train['source_id'].copy()
test_source_raw = test['source_id'].copy()
biome_col_raw = train['biome'].copy()
test_biome_raw = test['biome'].copy()

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Target skewness: {train[TARGET].skew():.4f}")

# ============================================================
# FEATURE ENGINEERING
# ============================================================
def feature_engineering(df):
    df = df.copy()

    # Drop zero-variance
    drop = ['sample_id', 'has_band_A_spectrum', 'sampling_depth_cm']
    df.drop(columns=[c for c in drop if c in df.columns], inplace=True)

    # Missing indicators (MNAR — missing itu sendiri informatif)
    df['band_B_missing']    = df['spectral_band_B_PC_1'].isna().astype(np.int8)
    df['latlon_missing']    = df['latitude'].isna().astype(np.int8)
    df['acidity_missing']   = df['property_acidity_index'].isna().astype(np.int8)
    df['cation_na_missing'] = df['cation_Na'].isna().astype(np.int8)
    df['cation_missing']    = df['cation_Ca'].isna().astype(np.int8)

    # Tekstur partikel tanah
    df['particle_ratio'] = df['property_particle_coarse'] / (df['property_particle_fine'] + 1e-6)
    df['particle_sum']   = df['property_particle_coarse'] + df['property_particle_fine']
    df['particle_diff']  = df['property_particle_coarse'] - df['property_particle_fine']

    # Kimia tanah — saturasi kation
    df['ca_saturation'] = df['cation_Ca'] / (df['cation_exchange_capacity'] + 1e-6)
    df['mg_saturation'] = df['cation_Mg'] / (df['cation_exchange_capacity'] + 1e-6)
    df['cation_sum']    = df['cation_Ca'].fillna(0) + df['cation_Mg'].fillna(0)

    # Log transform kation (distribusi skewed)
    for col in ['cation_Ca', 'cation_Mg', 'cation_exchange_capacity', 'cation_sum']:
        df[f'log_{col}'] = np.log1p(df[col].clip(lower=0))

    # Spektral Band A (selalu tersedia)
    ba = [f'spectral_band_A_PC_{i}' for i in range(1, 16)]
    df['band_A_magnitude'] = np.sqrt((df[ba] ** 2).sum(axis=1))
    df['band_A_mean']      = df[ba].mean(axis=1)
    df['band_A_std']       = df[ba].std(axis=1)

    # Spektral Band B (85% missing — biarkan NaN, LightGBM handle)
    bb = [f'spectral_band_B_PC_{i}' for i in range(1, 16)]
    df['band_B_magnitude'] = np.sqrt((df[bb] ** 2).sum(axis=1))
    df['band_B_mean']      = df[bb].mean(axis=1)

    # Geografis
    df['abs_latitude'] = df['latitude'].abs()

    # Encode categorical → category dtype (LightGBM native)
    cat_cols = [
        'source_id', 'has_band_B_spectrum', 'sampling_strategy',
        'geo_zone_macro', 'geo_zone_micro', 'geo_zone_meso',
        'land_cover_type', 'biome', 'parent_rock_type'
    ]
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype('category')

    return df

train = feature_engineering(train)
test  = feature_engineering(test)

# Target — log transform
y_raw = train[TARGET].copy()
y     = np.log1p(y_raw)

train.drop(columns=[TARGET], inplace=True)
test = test[train.columns]

# ============================================================
# OOF TARGET ENCODING — source_id & biome
# WAJIB OOF: jangan fit di seluruh train → leakage!
# source_id punya target mean beda 10x antar sumber
# ============================================================
kf_te       = KFold(n_splits=10, shuffle=True, random_state=SEED)
global_mean = y_raw.mean()

def oof_target_encode(col_train, col_test):
    """Target encoding dengan OOF — bebas data leakage."""
    oof_enc  = np.zeros(len(train))
    full_map = y_raw.groupby(col_train).mean()

    for trn_idx, val_idx in kf_te.split(train):
        fold_map = y_raw.iloc[trn_idx].groupby(col_train.iloc[trn_idx]).mean()
        oof_enc[val_idx] = col_train.iloc[val_idx].map(fold_map).fillna(global_mean).values

    test_enc = col_test.map(full_map).fillna(global_mean).values
    return oof_enc, test_enc

train['source_target_enc'], test['source_target_enc'] = \
    oof_target_encode(source_col_raw, test_source_raw)

train['biome_target_enc'], test['biome_target_enc'] = \
    oof_target_encode(biome_col_raw, test_biome_raw)

print(f"Total fitur: {train.shape[1]}")

# ============================================================
# TRAINING — LightGBM 5-Fold KFold
# ============================================================
lgbm_params = {
    'objective'        : 'regression',
    'metric'           : 'rmse',
    'n_estimators'     : 3000,
    'learning_rate'    : 0.02,
    'num_leaves'       : 127,
    'max_depth'        : -1,
    'min_child_samples': 20,
    'subsample'        : 0.8,
    'subsample_freq'   : 1,
    'colsample_bytree' : 0.6,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 1.0,
    'random_state'     : SEED,
    'verbose'          : -1,
    'n_jobs'           : -1,
}

kf    = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(kf.split(train))

oof  = np.zeros(len(train))
pred = np.zeros(len(test))
fold_rmses = []

print("\nTraining LightGBM...")
for fold_num, (trn_idx, val_idx) in enumerate(folds):
    X_trn, X_val = train.iloc[trn_idx], train.iloc[val_idx]
    y_trn, y_val = y.iloc[trn_idx],     y.iloc[val_idx]

    model = lgb.LGBMRegressor(**lgbm_params)
    model.fit(
        X_trn, y_trn,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(-1),
        ]
    )

    oof[val_idx] = model.predict(X_val)
    pred        += model.predict(test) / N_FOLDS

    fold_rmse = rmse(np.expm1(y_val), np.expm1(oof[val_idx]))
    fold_rmses.append(fold_rmse)
    print(f"  Fold {fold_num+1}: RMSE={fold_rmse:.4f} | iter={model.best_iteration_}")

oof_rmse = rmse(np.expm1(y), np.expm1(oof))
print(f"\nOOF RMSE: {oof_rmse:.4f} +/- {np.std(fold_rmses):.4f}")

# ============================================================
# GENERATE SUBMISSION
# ============================================================
pred_final = np.clip(np.expm1(pred), a_min=0, a_max=None)

submission = pd.DataFrame({
    'sample_id'               : test_ids,
    'property_organic_content': pred_final,
})

# Sanity check
assert len(submission) == 2670
assert submission['property_organic_content'].isna().sum() == 0

submission.to_csv('submission.csv', index=False)
print(f"\nsubmission.csv saved")
print(submission.describe())

# Simpan OOF untuk Research Questions
oof_df = pd.DataFrame({
    'sample_id': train_ids,
    'y_true'   : y_raw.values,
    'oof_pred' : np.expm1(oof),
})
oof_df.to_csv('oof_predictions.csv', index=False)
print("oof_predictions.csv saved (untuk Research Questions)")

Train: (11210, 52) | Test: (2670, 51)
Target skewness: 2.0278
Total fitur: 71

Training LightGBM...
  Fold 1: RMSE=11.1487 | iter=762
  Fold 2: RMSE=10.8818 | iter=1050
  Fold 3: RMSE=11.2780 | iter=522
  Fold 4: RMSE=11.8992 | iter=588
  Fold 5: RMSE=12.7932 | iter=719

OOF RMSE: 11.6203 +/- 0.6836

submission.csv saved
       property_organic_content
count               2670.000000
mean                  33.092275
std                   19.456696
min                    5.827297
25%                   19.386934
50%                   26.914274
75%                   41.782728
max                  156.892223
oof_predictions.csv saved (untuk Research Questions)
